In [1]:
# Install dependencies as needed:
# pip install kagglehub[pandas-datasets]
import kagglehub
from kagglehub import KaggleDatasetAdapter

# Set the path to the file you'd like to load
file_path = "news_article_dataset.tsv"

# Load the latest version
df = kagglehub.load_dataset(
  KaggleDatasetAdapter.PANDAS,
  "articoder/news-dataset-for-news-bias-analysis",
  file_path,
  # Provide any additional arguments like
  # sql_query or pandas_kwargs. See the
  # documenation for more information:
  # https://github.com/Kaggle/kagglehub/blob/main/README.md#kaggledatasetadapterpandas
)

print("First 5 records:", df.head())

<ipython-input-1-c213d0f70412>:10: DeprecationWarning: load_dataset is deprecated and will be removed in a future version.
  df = kagglehub.load_dataset(


100%|██████████| 23.2M/23.2M [00:00<00:00, 50.5MB/s]


First 5 records:    Unnamed: 0  Title of Headline Roundup description       Topics        Date  \
0        3513           Romney in London         NaN    Elections  2012-07-26   
1        3512     Romney's Overseas Trip         NaN    Elections  2012-08-01   
2        3511  Biden Comment Controversy         NaN    Elections  2012-08-16   
3        3510               Romney Taxes         NaN    Elections  2012-08-17   
4        3509  Skinnydipping Congressman         NaN  US Congress  2012-08-20   

                          url_story  \
0              /story/romney-london   
1      /story/romneys-overseas-trip   
2  /story/biden-comment-controversy   
3               /story/romney-taxes   
4  /story/skinnydipping-congressman   

                                    left_story_title  \
0                      Romney's Olympics false start   
1  Was Romney's trip 'a great success' or gaffe-f...   
2            Mission Impossible: Managing Joe Biden    
3  Obama Campaign Wants Romney To Rel

In [2]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8478 entries, 0 to 8477
Data columns (total 15 columns):
 #   Column                     Non-Null Count  Dtype 
---  ------                     --------------  ----- 
 0   Unnamed: 0                 8478 non-null   int64 
 1   Title of Headline Roundup  8478 non-null   object
 2   description                4963 non-null   object
 3   Topics                     8478 non-null   object
 4   Date                       8478 non-null   object
 5   url_story                  8478 non-null   object
 6   left_story_title           8436 non-null   object
 7   center_story_title         7706 non-null   object
 8   right_story_title          8381 non-null   object
 9   right_story_url            8381 non-null   object
 10  right_story_text           8375 non-null   object
 11  center_story_url           7706 non-null   object
 12  center_story_text          7700 non-null   object
 13  left_story_url             8436 non-null   object
 14  left_sto

In [7]:
import pandas as pd

# Set pandas to display all rows
pd.set_option('display.max_rows', None)

In [8]:
print(df['Topics'].value_counts())

Topics
Politics                                744
Elections                               615
Economy and Jobs                        462
Coronavirus                             414
World                                   387
Middle East                             334
Donald Trump                            294
Immigration                             289
2024 Presidential Election              283
White House                             226
Supreme Court                           208
Violence in America                     203
Joe Biden                               190
Justice                                 152
Defense and Security                    138
Russia                                  133
Technology                              128
Media Industry                          126
Criminal Justice                        112
Healthcare                              110
General News                            106
Abortion                                104
Gun Control and Gun Right

In [ ]:
import pandas as pd

# Assuming your DataFrame is already loaded into `df`

# Rename relevant columns for consistency
df = df.rename(columns={
    'Title of Headline Roundup': 'roundup_title',
    'Topics': 'topics',
    'left_story_title': 'left_title',
    'left_story_text': 'left_text',
    'center_story_title': 'center_title',
    'center_story_text': 'center_text',
    'right_story_title': 'right_title',
    'right_story_text': 'right_text'
})

# Create Left Bias Subset
left_df = df[['roundup_title', 'topics', 'left_title', 'left_text']].dropna(subset=['left_title', 'left_text']).copy()
left_df['bias_label'] = 'left'
left_df.columns = ['roundup_title', 'topics', 'story_title', 'story_text', 'bias_label']

# Create Center Bias Subset
center_df = df[['roundup_title', 'topics', 'center_title', 'center_text']].dropna(subset=['center_title', 'center_text']).copy()
center_df['bias_label'] = 'center'
center_df.columns = ['roundup_title', 'topics', 'story_title', 'story_text', 'bias_label']

# Create Right Bias Subset
right_df = df[['roundup_title', 'topics', 'right_title', 'right_text']].dropna(subset=['right_title', 'right_text']).copy()
right_df['bias_label'] = 'right'
right_df.columns = ['roundup_title', 'topics', 'story_title', 'story_text', 'bias_label']

# Concatenate all subsets
final_df = pd.concat([left_df, center_df, right_df], ignore_index=True)

# Shuffle rows (optional)
final_df = final_df.sample(frac=1, random_state=42).reset_index(drop=True)

# Preview
print(final_df.head())
print("\nClass balance:\n", final_df['bias_label'].value_counts())


                                       roundup_title             topics  \
0  Trump Considers Revoking Obama Intel Officials...  National Security   
1  How Political Polarization Paved the Way for t...       Polarization   
2  Venezuela Arrests 3 Americans Involved in Alle...       The Americas   
3  Abortion Bans to Take Effect in Florida, Missi...           Abortion   
4    US and China Agree to Suspend New Trade Tariffs     Foreign Policy   

                                         story_title  \
0  President Trump considers revoking security cl...   
1  Whether Robert Fico survives and resumes offic...   
2  US denies claim CIA plotted to kill Venezuela ...   
3  Court won't block Mississippi's abortion "trig...   
4  US-China trade war: Deal agreed to suspend new...   

                                          story_text bias_label  
0  President Donald Trump is exploring "mechanism...     center  
1  A few years after the dissolution of Czechoslo...       left  
2  The United 

In [ ]:
# Sort the combined dataframe by roundup title
final_df = final_df.sort_values(by='roundup_title').reset_index(drop=True)

final_df.head(50)

,roundup_title,topics,story_title,story_text,bias_label
0,"""Dream"" 50 Years Later",Civil Rights,Thousands Gather In D.C. To Mark 1963 Civil Ri...,People are assembling on the National Mall to ...,center
1,"""Dream"" 50 Years Later",Civil Rights,Remembering My Uncle's 'Dream',"Fifty years ago, a valiant group of people fro...",right
2,"""Dream"" 50 Years Later",Civil Rights,March In Washington To Continue Focus On Civil...,Alice Long planned months ago to use vacation ...,left
3,"""Good Shutdown"" in September?",Politics,"President Trump Calls for a ""Good Shutdown"" in...",President Donald Trump made a bold statement o...,right
4,"""Good Shutdown"" in September?",Politics,Trump's frustration with budget compromise has...,Congress looks set to enact a bipartisan spend...,left
5,"""Good Shutdown"" in September?",Politics,Trump: US ‘needs a good shutdown’,"President Trump on Tuesday called for a ""good ...",center
6,"""Skinny Repeal"" Motions Fails",US Senate,Obamacare repeal is dead for now. What could t...,The Senate’s effort to repeal and replace Obam...,center
7,"""Skinny Repeal"" Motions Fails",US Senate,Why Senate Republicans couldn’t repeal Obamacare,"In a stunning turn, Senate Republicans — in th...",left
8,"""Skinny Repeal"" Motions Fails",US Senate,Senate Fails To Pass Motion To Proceed On ‘Ski...,Senate Majority Leader Mitch McConnell failed ...,right
9,"""Trump University"" Documents Unsealed",Elections,'Trump University' Documents Put On Display Ag...,A federal judge released hundreds of pages of ...,center


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report

# Combine title and text fields
final_df['combined_text'] = final_df['story_title'].fillna('') + ' ' + final_df['story_text'].fillna('')

# Define features and target
X = final_df['combined_text']
y = final_df['bias_label']

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(X, y, stratify=y, random_state=42)

# Build pipeline: TF-IDF + Logistic Regression
pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(
        stop_words='english',
        ngram_range=(1, 2),      # unigrams + bigrams
        max_features=5000        # top 5000 features
    )),
    ('clf', LogisticRegression(
        max_iter=1000,
        class_weight='balanced'  # handle label imbalance
    ))
])

# Train the model
pipeline.fit(X_train, y_train)

# Predict and evaluate
y_pred = pipeline.predict(X_test)
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

      center       0.38      0.39      0.38      1925
        left       0.43      0.41      0.42      2108
       right       0.46      0.47      0.46      2094

    accuracy                           0.42      6127
   macro avg       0.42      0.42      0.42      6127
weighted avg       0.42      0.42      0.42      6127



In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report

binary_df = final_df[final_df['bias_label'].isin(['left', 'right'])].copy()

# Combine title and text fields
binary_df['combined_text'] = binary_df['story_title'].fillna('') + ' ' + binary_df['story_text'].fillna('')

# Define features and target
X = binary_df['combined_text']
y = binary_df['bias_label']

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(X, y, stratify=y, random_state=42)

# Build pipeline: TF-IDF + Logistic Regression
pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(
        stop_words='english',
        ngram_range=(1, 2),      # unigrams + bigrams
        max_features=5000        # top 5000 features
    )),
    ('clf', LogisticRegression(
        max_iter=1000,
        class_weight='balanced'  # handle label imbalance
    ))
])

# Train the model
pipeline.fit(X_train, y_train)

# Predict and evaluate
y_pred = pipeline.predict(X_test)
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

        left       0.62      0.60      0.61      2108
       right       0.61      0.62      0.62      2094

    accuracy                           0.61      4202
   macro avg       0.61      0.61      0.61      4202
weighted avg       0.61      0.61      0.61      4202



In [ ]:
# Get feature names
feature_names = pipeline.named_steps['tfidf'].get_feature_names_out()
coefs = pipeline.named_steps['clf'].coef_[0]

# Top left-leaning words
top_left = sorted(zip(coefs, feature_names))[:20]
print("Top left-leaning features:")
for weight, word in top_left:
    print(f"{word}: {weight:.4f}")

# Top right-leaning words
top_right = sorted(zip(coefs, feature_names), reverse=True)[:20]
print("\nTop right-leaning features:")
for weight, word in top_right:
    print(f"{word}: {weight:.4f}")

Top left-leaning features:
matters: -3.4364
cnn: -2.2649
donald trump: -2.2413
donald: -2.2125
decades: -2.0824
familiar: -1.9102
far: -1.8969
mr: -1.8283
driving: -1.8126
joe biden: -1.7907
president joe: -1.7563
months: -1.7387
driving news: -1.6967
live: -1.6791
united states: -1.6611
care: -1.6130
big picture: -1.5628
joe: -1.5582
soon: -1.5575
important: -1.5563

Top right-leaning features:
president trump: 3.5738
reportedly: 3.0594
illegal: 2.3412
fox news: 2.2108
reported: 2.1085
obamacare: 2.0912
doj: 2.0781
left: 2.0228
terrorists: 2.0017
biden: 1.9353
illegal immigrants: 1.8914
terror: 1.8647
press: 1.8619
told fox: 1.8466
president biden: 1.8428
ca: 1.8192
revealed: 1.8138
dems: 1.7888
slams: 1.7702
media: 1.7584


In [ ]:
# Install and import dependencies
# Run this once in your environment if needed:
# !pip install spacy
# !python -m spacy download en_core_web_sm

import pandas as pd
import spacy
import re
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report

# Load spaCy English model (disable unneeded components for speed)
nlp = spacy.load("en_core_web_sm", disable=["parser", "ner"])

# Text cleaning function
def clean_text(text):
    text = text.lower()                           # Lowercase
    text = re.sub(r'\d+', '', text)               # Remove digits
    text = re.sub(r'[^\w\s]', '', text)           # Remove punctuation
    text = re.sub(r'\s+', ' ', text).strip()      # Remove extra whitespace
    doc = nlp(text)                               # Lemmatize and remove stopwords
    tokens = [token.lemma_ for token in doc if not token.is_stop]
    return " ".join(tokens)

# Filter only left and right classes
binary_df = final_df[final_df['bias_label'].isin(['left', 'right'])].copy()

# Combine story_title + story_text
binary_df['combined_text'] = binary_df['story_title'].fillna('') + ' ' + binary_df['story_text'].fillna('')

# Apply text cleaning
binary_df['clean_text'] = binary_df['combined_text'].apply(clean_text)

# Features and target
X = binary_df['clean_text']
y = binary_df['bias_label']

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, stratify=y, random_state=42)

# TF-IDF + Logistic Regression pipeline
pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(
        stop_words='english',
        ngram_range=(1, 2),
        max_features=5000
    )),
    ('clf', LogisticRegression(
        max_iter=1000,
        class_weight='balanced'
    ))
])

# Train the model
pipeline.fit(X_train, y_train)

# Predict and evaluate
y_pred = pipeline.predict(X_test)
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

        left       0.61      0.60      0.60      2108
       right       0.60      0.62      0.61      2094

    accuracy                           0.61      4202
   macro avg       0.61      0.61      0.61      4202
weighted avg       0.61      0.61      0.61      4202



In [ ]:
# If needed, install dependencies:
# !pip install spacy xgboost
# !python -m spacy download en_core_web_sm

import pandas as pd
import spacy
import re
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from xgboost import XGBClassifier
from sklearn.preprocessing import LabelEncoder

# Load spaCy
nlp = spacy.load("en_core_web_sm", disable=["parser", "ner"])

# Cleaning function
def clean_text(text):
    text = text.lower()
    text = re.sub(r'\d+', '', text)
    text = re.sub(r'[^\w\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    doc = nlp(text)
    tokens = [token.lemma_ for token in doc if not token.is_stop and token.is_alpha]
    return " ".join(tokens)

# Filter to left and right
binary_df = final_df[final_df['bias_label'].isin(['left', 'right'])].copy()

# Combine title and text
binary_df['combined_text'] = binary_df['story_title'].fillna('') + ' ' + binary_df['story_text'].fillna('')

# Clean text
binary_df['clean_text'] = binary_df['combined_text'].apply(clean_text)

# Encode labels: left → 0, right → 1
le = LabelEncoder()
y = le.fit_transform(binary_df['bias_label'])  # ['left', 'right'] → [0, 1]

# Features
X = binary_df['clean_text']

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, stratify=y, random_state=42)

# TF-IDF Vectorizer
vectorizer = TfidfVectorizer(
    stop_words='english',
    ngram_range=(1, 2),
    max_features=5000
)

# Fit and transform
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

# Train XGBoost
xgb_model = XGBClassifier(use_label_encoder=False, eval_metric='logloss')
xgb_model.fit(X_train_tfidf, y_train)

# Predict
y_pred = xgb_model.predict(X_test_tfidf)

# Decode predictions back to labels
y_test_labels = le.inverse_transform(y_test)
y_pred_labels = le.inverse_transform(y_pred)

# Classification report
print(classification_report(y_test_labels, y_pred_labels))

/usr/local/lib/python3.11/dist-packages/xgboost/core.py:158: UserWarning: [17:32:48] WARNING: /workspace/src/learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)


              precision    recall  f1-score   support

        left       0.60      0.62      0.61      2108
       right       0.61      0.59      0.60      2094

    accuracy                           0.61      4202
   macro avg       0.61      0.61      0.61      4202
weighted avg       0.61      0.61      0.61      4202



In [ ]:
import pandas as pd
import spacy
import re
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Bidirectional, LSTM, Dense, Dropout
from tensorflow.keras.utils import to_categorical

# Load spaCy
nlp = spacy.load("en_core_web_sm", disable=["parser", "ner"])

# Cleaning function
def clean_text(text):
    text = text.lower()
    text = re.sub(r'\d+', '', text)
    text = re.sub(r'[^\w\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    doc = nlp(text)
    tokens = [token.lemma_ for token in doc if not token.is_stop and token.is_alpha]
    return " ".join(tokens)

# Filter binary classes
binary_df = final_df[final_df['bias_label'].isin(['left', 'right'])].copy()
binary_df['combined_text'] = binary_df['story_title'].fillna('') + ' ' + binary_df['story_text'].fillna('')
binary_df['clean_text'] = binary_df['combined_text'].apply(clean_text)

# Encode labels
le = LabelEncoder()
binary_df['label_encoded'] = le.fit_transform(binary_df['bias_label'])  # left = 0, right = 1

# Split data
X_train, X_test, y_train, y_test = train_test_split(
    binary_df['clean_text'],
    binary_df['label_encoded'],
    stratify=binary_df['label_encoded'],
    test_size=0.2,
    random_state=42
)

# Tokenize text
vocab_size = 10000
max_length = 300
tokenizer = Tokenizer(num_words=vocab_size, oov_token="<OOV>")
tokenizer.fit_on_texts(X_train)

X_train_seq = tokenizer.texts_to_sequences(X_train)
X_test_seq = tokenizer.texts_to_sequences(X_test)

X_train_pad = pad_sequences(X_train_seq, maxlen=max_length, padding='post', truncating='post')
X_test_pad = pad_sequences(X_test_seq, maxlen=max_length, padding='post', truncating='post')

# Convert labels to categorical (optional, for softmax)
# y_train_cat = to_categorical(y_train)
# y_test_cat = to_categorical(y_test)

# Define the model
model = Sequential([
    Embedding(input_dim=vocab_size, output_dim=128, input_length=max_length),
    Bidirectional(LSTM(64, return_sequences=False)),
    Dropout(0.3),
    Dense(32, activation='relu'),
    Dense(1, activation='sigmoid')  # For binary classification
])

model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])

# Train the model
history = model.fit(X_train_pad, y_train, epochs=5, batch_size=32, validation_split=0.1)

# Predict and evaluate
y_pred_prob = model.predict(X_test_pad)
y_pred = (y_pred_prob > 0.5).astype(int).flatten()

# Report
print(classification_report(y_test, y_pred, target_names=le.classes_))

Epoch 1/5


/usr/local/lib/python3.11/dist-packages/keras/src/layers/core/embedding.py:90: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


379/379 ━━━━━━━━━━━━━━━━━━━━ 168s 416ms/step - accuracy: 0.4994 - loss: 0.6936 - val_accuracy: 0.5509 - val_loss: 0.6835
Epoch 2/5
379/379 ━━━━━━━━━━━━━━━━━━━━ 140s 371ms/step - accuracy: 0.6487 - loss: 0.6344 - val_accuracy: 0.5346 - val_loss: 0.7034
Epoch 3/5
379/379 ━━━━━━━━━━━━━━━━━━━━ 139s 366ms/step - accuracy: 0.7847 - loss: 0.4736 - val_accuracy: 0.5561 - val_loss: 0.8074
Epoch 4/5
379/379 ━━━━━━━━━━━━━━━━━━━━ 142s 367ms/step - accuracy: 0.8883 - loss: 0.2874 - val_accuracy: 0.5673 - val_loss: 1.0202
Epoch 5/5
379/379 ━━━━━━━━━━━━━━━━━━━━ 143s 376ms/step - accuracy: 0.9394 - loss: 0.1681 - val_accuracy: 0.5613 - val_loss: 1.3782
106/106 ━━━━━━━━━━━━━━━━━━━━ 9s 79ms/step
              precision    recall  f1-score   support

        left       0.55      0.59      0.57      1686
       right       0.55      0.52      0.54      1675

    accuracy                           0.55      3361
   macro avg       0.55      0.55      0.55      3361
weighted avg       0.55      0.55      0.

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from datasets import Dataset
from transformers import BertTokenizerFast, BertForSequenceClassification, Trainer, TrainingArguments
from sklearn.metrics import classification_report
import numpy as np

# 1. Prepare the dataset
# Filter to binary labels
binary_df = final_df[final_df['bias_label'].isin(['left', 'right'])].copy()
binary_df['combined_text'] = binary_df['story_title'].fillna('') + ' ' + binary_df['story_text'].fillna('')

# Encode labels: left=0, right=1
le = LabelEncoder()
binary_df['label'] = le.fit_transform(binary_df['bias_label'])

# Train/test split
train_df, test_df = train_test_split(binary_df[['combined_text', 'label']], stratify=binary_df['label'], random_state=42)

# HuggingFace Datasets format
train_dataset = Dataset.from_pandas(train_df)
test_dataset = Dataset.from_pandas(test_df)

# 2. Load tokenizer and tokenize data
tokenizer = BertTokenizerFast.from_pretrained('bert-base-uncased')

def tokenize(example):
    return tokenizer(
        example['combined_text'],
        padding='max_length',
        truncation=True,
        max_length=512
    )

train_dataset = train_dataset.map(tokenize, batched=True)
test_dataset = test_dataset.map(tokenize, batched=True)

# Set format for PyTorch
train_dataset.set_format(type='torch', columns=['input_ids', 'attention_mask', 'label'])
test_dataset.set_format(type='torch', columns=['input_ids', 'attention_mask', 'label'])

# 3. Load model
model = BertForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=2)

# 4. Training arguments
training_args = TrainingArguments(
    output_dir='./results',
    evaluation_strategy='epoch',
    save_strategy='epoch',
    logging_strategy='epoch',
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model='eval_loss'
)

# 5. Metrics function
def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    preds = np.argmax(predictions, axis=1)
    report = classification_report(labels, preds, output_dict=True, zero_division=0)
    return {
        "precision": report["weighted avg"]["precision"],
        "recall": report["weighted avg"]["recall"],
        "f1": report["weighted avg"]["f1-score"],
        "accuracy": report["accuracy"]
    }

# 6. Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics
)

# 7. Train
trainer.train()

# 8. Final evaluation
predictions = trainer.predict(test_dataset)
preds = np.argmax(predictions.predictions, axis=1)
print(classification_report(test_df['label'], preds, target_names=le.classes_))

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

...

Map:   0%|          | 0/4202 [00:00<?, ? examples/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


TypeError: TrainingArguments.__init__() got an unexpected keyword argument 'evaluation_strategy'